# ROGII - fix scale mismatch: reconstruct absolute TVT from delta-based ridge_oof_pred/target


In [ ]:
"""Quick fix: their_ridge_oof_773.pkl's 'target'/'ridge_oof_pred' turned out to be DELTAS from
last_known_tvt (confirmed: both have small-magnitude stats, mean~1.6 std~15, not absolute TVT ~11000+),
not absolute TVT as first assumed -- this caused a catastrophic scale-mismatch NaN in GRU training
(residual = absolute_true - delta_pred nonsensically subtracts a ~11000 number from a ~50 number).
Fix: compute last_known_tvt per well (trivial -- just the last known TVT_input value, no heavy PF/GBM
recompute needed) and reconstruct proper ABSOLUTE sp45-equivalent = last_known_tvt + ridge_oof_pred,
absolute true = last_known_tvt + target (should match raw TVT exactly, sanity-checked).
"""
import glob, os, pickle, numpy as np, pandas as pd

_c = glob.glob('/kaggle/input/**/*__horizontal_well.csv', recursive=True)
_t = [p for p in _c if 'train' in p.lower()]
_c = _t if _t else _c
TRAIN_DIR = os.path.dirname(_c[0]) if _c else 'd:/ROGII/data/train'
_assets = glob.glob('/kaggle/input/**/their_ridge_oof_773.pkl', recursive=True)
ASSET_DIR = os.path.dirname(_assets[0]) if _assets else '.'
print(f'TRAIN_DIR={TRAIN_DIR}  ASSET_DIR={ASSET_DIR}', flush=True)

d = pickle.load(open(f'{ASSET_DIR}/their_ridge_oof_773.pkl', 'rb'))
print('loaded', len(d), 'rows,', d['well'].nunique(), 'wells', flush=True)

last_known_by_well = {}
for wid in d['well'].unique():
    hw = pd.read_csv(f'{TRAIN_DIR}/{wid}__horizontal_well.csv', usecols=['TVT_input'])
    kn = hw['TVT_input'].dropna()
    if len(kn) == 0:
        last_known_by_well[wid] = np.nan
        continue
    last_known_by_well[wid] = float(kn.iloc[-1])

d['last_known_tvt'] = d['well'].map(last_known_by_well)
n_missing = d['last_known_tvt'].isna().sum()
print('rows with missing last_known_tvt:', n_missing, flush=True)

d['abs_ridge_oof_pred'] = d['last_known_tvt'] + d['ridge_oof_pred']
d['abs_true'] = d['last_known_tvt'] + d['target']

# sanity check against the raw TVT column directly
sample_wids = list(d['well'].unique()[:5])
for wid in sample_wids:
    hw = pd.read_csv(f'{TRAIN_DIR}/{wid}__horizontal_well.csv', usecols=['TVT_input', 'TVT'])
    ev_mask = hw['TVT_input'].isna().values
    raw_true = hw['TVT'].values[ev_mask]
    sub = d[d['well'] == wid].sort_values('id', key=lambda s: s.str.rsplit('_', n=1).str[-1].astype(int))
    diff = np.abs(sub['abs_true'].to_numpy() - raw_true[:len(sub)])
    print(f'{wid}: max diff abs_true vs raw TVT = {diff.max():.6f}', flush=True)

rmse_abs = float(np.sqrt(np.mean((d['abs_true'] - d['abs_ridge_oof_pred']) ** 2)))
print(f'pooled RMSE (absolute scale, sanity recheck): {rmse_abs:.4f}', flush=True)

pickle.dump(d, open('their_ridge_oof_773_fixed.pkl', 'wb'))
print('saved their_ridge_oof_773_fixed.pkl', flush=True)

